# Học hiệu chỉnh khung biển số
Thử hồi quy khung theo ý tưởng [R-CNN](https://arxiv.org/abs/1311.2524) với đặc trưng HOG và nhãn của dự án. Giữ SVM đã huấn luyện trên train của dự án. Ridge học độ dịch và độ co giãn của khung, rồi NMS loại các khung trùng. Chạy trên CPU từ gốc repo sau khi hoàn tất `optimize-svm-detector.ipynb`.

In [ ]:
import csv
import json
import random
from pathlib import Path

import cv2
import numpy as np
from sklearn.linear_model import Ridge

from src.detector import (
    PlateDetector, box_transform, iou, proposal_features, propose_ensemble, refine_box,
    suppress_overlaps,
)

DATASET_OVERRIDE = None
SMOKE_RUN = False
SEED = 42
IOU_THRESHOLD = 0.5
OUTPUT_DIR = Path('outputs/svm_refined_smoke' if SMOKE_RUN else 'outputs/svm_refined')

dataset_name = 'License Plate Detection Dataset'
roots = [
    Path(dataset_name),
    Path('../CuoiKy') / dataset_name,
]
kaggle_input = Path('/kaggle/input')
if kaggle_input.is_dir():
    roots.extend(sorted(kaggle_input.rglob(dataset_name)))
if DATASET_OVERRIDE is not None:
    roots = [Path(DATASET_OVERRIDE)]
root = next((p for p in roots if (p / 'images').is_dir() and (p / 'labels').is_dir()), None)
if root is None:
    raise FileNotFoundError('Dat DATASET_OVERRIDE den thu muc co images/ va labels/.')
print(root.resolve())

BASE_CONFIG_PATH = Path('outputs/svm_optimized/detector_config.json')
baseline = PlateDetector(BASE_CONFIG_PATH)
base_config = baseline.config

def proposals(gray):
    return propose_ensemble(gray, base_config['ensemble'], base_config['max_side'])

## 1. Nhãn và tập dữ liệu
Tập train dùng để học trọng số. Tập val chọn hệ số regularization `C` và ngưỡng SVM. Tập test dùng để so sánh cuối cùng. Danh sách loại trừ là các bản trùng pixel đã xác định trong EDA.

In [ ]:
EXACT_DUPLICATES = {
    'train': {'quandoi64.jpg', 'quandoi92.jpg', 'quandoi99.jpg'},
    'val': set(),
    'test': {'quandoi75.jpg'},
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def image_paths(directory):
    return sorted(p for p in directory.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)

def split_paths(split):
    paths = image_paths(root / 'images' / split)
    paths = [p for p in paths if p.name not in EXACT_DUPLICATES[split]]
    if SMOKE_RUN:
        paths = random.Random(SEED).sample(paths, min(18, len(paths)))
    if not paths:
        raise ValueError(f'Tap {split} rong.')
    return paths

def read_gray(path):
    gray = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f'Khong doc duoc: {path}')
    return gray

def ground_truth(path, split, image_shape):
    height, width = image_shape
    relative = path.relative_to(root / 'images' / split).with_suffix('.txt')
    label_path = root / 'labels' / split / relative
    boxes = []
    for row in label_path.read_text(encoding='utf-8-sig').splitlines():
        if not row.strip():
            continue
        label, x, y, w, h = map(float, row.split())
        if label != 0 or w <= 0 or h <= 0:
            raise ValueError(f'Nhan khong hop le: {label_path}')
        box = (
            max(0, (x - w / 2) * width),
            max(0, (y - h / 2) * height),
            min(width, (x + w / 2) * width),
            min(height, (y + h / 2) * height),
        )
        if box[0] >= box[2] or box[1] >= box[3]:
            raise ValueError(f'Khung nam ngoai anh: {label_path}')
        boxes.append(box)
    return boxes

train_paths = split_paths('train')
val_paths = split_paths('val')
test_paths = split_paths('test')
print(f'Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}')

## Học điều chỉnh khung trên train
Mỗi vùng ứng viên có IoU ≥ 0,3 được ghép với nhãn gần nhất. Hồi quy dự đoán độ lệch tâm và log tỷ lệ rộng/cao so với khung nhãn.

In [ ]:
features, targets = [], []
for path in train_paths:
    gray = read_gray(path)
    truth = ground_truth(path, 'train', gray.shape)
    if not truth:
        continue
    for box, _ in proposals(gray):
        target = max(truth, key=lambda ref: iou(box, ref))
        if iou(box, target) >= 0.3:
            features.append(proposal_features(gray, box))
            targets.append(box_transform(box, target))

X_train = np.stack(features)
regressor = Ridge(alpha=10.0).fit(X_train, np.asarray(targets, dtype=np.float32))
print(f'Regression samples: {len(features)}')
del features, targets, X_train

## Chọn cấu hình trên val
Đo ảnh hưởng của mức chỉnh khung, ngưỡng điểm SVM và ngưỡng NMS. Ưu tiên các cấu hình tăng cả precision và recall trên val. Nếu không có, chọn F1 cao nhất. Test được dùng để đối chiếu sau khi cấu hình đã chốt.

In [ ]:
def match_counts(predictions, truth):
    available = set(range(len(truth)))
    matched = 0
    for box, _ in predictions:
        if not available:
            break
        index = max(sorted(available), key=lambda i: iou(box, truth[i]))
        if iou(box, truth[index]) >= IOU_THRESHOLD:
            available.remove(index)
            matched += 1
    return np.array([matched, len(predictions) - matched, len(truth) - matched])

def metrics(counts):
    tp, fp, fn = map(int, counts)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)

def collect_split(paths, split):
    for path in paths:
        gray = read_gray(path)
        truth = ground_truth(path, split, gray.shape)
        boxes = proposals(gray)
        if not boxes:
            yield truth, [], []
            continue
        features = np.stack([proposal_features(gray, box) for box, _ in boxes])
        scores = features @ baseline.weights + baseline.intercept
        transforms = regressor.predict(features)
        yield truth, gray.shape, list(zip([box for box, _ in boxes], transforms, scores))

def refine_candidates(shape, candidates, blend):
    return [(refine_box(box, delta * blend, shape) if blend else box, float(score))
            for box, delta, score in candidates]

def filter_candidates(candidates, threshold, overlap):
    return suppress_overlaps([(box, score) for box, score in candidates if score >= threshold],
                             overlap, base_config['max_boxes'])

val_records = list(collect_split(val_paths, 'val'))
base_threshold, base_nms = base_config['threshold'], base_config['nms_iou']
base_counts = np.zeros(3, dtype=int)
for truth, shape, candidates in val_records:
    base_counts += match_counts(filter_candidates(refine_candidates(shape, candidates, 0),
                                                  base_threshold, base_nms), truth)
base_val = metrics(base_counts)

val_rows = []
for blend in (0.0, 0.35, 0.65, 1.0):
    adjusted = [(truth, refine_candidates(shape, candidates, blend))
                for truth, shape, candidates in val_records]
    for threshold in (0.75, 0.85, 0.95, 1.0, 1.05, 1.15, 1.25):
        for overlap in (0.30, 0.38, 0.45, 0.55):
            counts = np.zeros(3, dtype=int)
            for truth, candidates in adjusted:
                counts += match_counts(filter_candidates(candidates, threshold, overlap), truth)
            val_rows.append(dict(blend=blend, threshold=threshold, nms_iou=overlap, **metrics(counts)))

improved = [row for row in val_rows if row['precision'] > base_val['precision']
            and row['recall'] > base_val['recall']]
selected = max(improved or val_rows, key=lambda row: (row['f1'], row['precision']))
print('Baseline val:', base_val)
print('Selected val:', selected, '| improved both:', bool(improved))

## Đánh giá test và lưu mô hình
So sánh hai cấu hình trên cùng ảnh và cùng IoU. Các tệp đầu ra gồm trọng số SVM, hệ số hồi quy khung và bảng kết quả.

In [ ]:
before, after = np.zeros(3, dtype=int), np.zeros(3, dtype=int)
for truth, shape, candidates in collect_split(test_paths, 'test'):
    original = refine_candidates(shape, candidates, 0)
    before += match_counts(filter_candidates(original, base_threshold, base_nms), truth)
    refined = refine_candidates(shape, candidates, selected['blend'])
    after += match_counts(filter_candidates(refined, selected['threshold'], selected['nms_iou']), truth)

test_rows = [dict(method='ensemble_svm', **metrics(before)),
             dict(method='ensemble_svm_refined', **metrics(after))]
for row in test_rows:
    print(row)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, rows in [('validation.csv', val_rows), ('test.csv', test_rows)]:
    with (OUTPUT_DIR / name).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
np.savez(OUTPUT_DIR / 'svm_weights.npz', weights=baseline.weights, intercept=baseline.intercept,
         box_weights=regressor.coef_.T, box_intercept=regressor.intercept_)
artifact = dict(base_config, threshold=selected['threshold'], nms_iou=selected['nms_iou'],
                box_blend=selected['blend'], smoke_run=SMOKE_RUN,
                train_images=len(train_paths), val_images=len(val_paths), test_images=len(test_paths))
artifact_path = OUTPUT_DIR / 'detector_config.json'
artifact_path.write_text(json.dumps(artifact, indent=2) + '\n', encoding='utf-8')

## 5. Kiểm tra nạp mô hình
Nạp lại file vừa xuất bằng lớp inference của dự án và so với kết quả SVM trên một số ảnh test.

In [ ]:
detector = PlateDetector(artifact_path)
for path, (_, shape, candidates) in zip(test_paths[:10], collect_split(test_paths[:10], 'test')):
    refined = refine_candidates(shape, candidates, selected['blend'])
    expected = filter_candidates(refined, selected['threshold'], selected['nms_iou'])
    actual = detector.predict(read_gray(path))
    np.testing.assert_allclose(np.asarray([box for box, _ in actual]),
                               np.asarray([box for box, _ in expected]), atol=1e-4)
print('Saved model verified on 10 images:', OUTPUT_DIR.resolve())